## Exercises on Exploration vs Exploitation (Multi-Armed Bandits)

These paper-and-pencil exercises reinforce Chapter 04: incremental action-value estimation, the sample-average and constant step-size updates, the exponential-recency weighting, the stochastic-approximation convergence conditions, $\varepsilon$-greedy action probabilities, and the UCB exploration bonus. In a $k$-armed bandit the value of an action is $q(a)=\mathbb{E}[R_t\mid A_t=a]$ and $Q_t(a)$ is its estimate.

### Exercise 4.1 — The incremental sample-average update

A single bandit arm returns the reward sequence $R_1,\dots,R_5 = 1,\,0,\,2,\,4,\,3$. Starting from $Q_1=0$:

1. Derive the incremental update rule for the sample average.
2. Apply it step by step to obtain $Q_2,\dots,Q_6$ and check the final value equals the plain average.

**Step 1 — Derive the incremental form.** The sample average after $n$ rewards is $Q_{n+1}=\frac1n\sum_{i=1}^n R_i$. Separating the last term:

$\displaystyle Q_{n+1} = \frac1n\Big(R_n+\sum_{i=1}^{n-1}R_i\Big) = \frac1n\big(R_n+(n-1)Q_n\big) = Q_n + \frac1n\big(R_n - Q_n\big).$

This is the general form **NewEstimate ← OldEstimate + StepSize·(Target − OldEstimate)** with step size $\tfrac1n$.

**Step 2 — Iterate** (target = the new reward $R_n$):

| $n$ | $R_n$ | update | $Q_{n+1}$ |
|---|---|---|---|
| 1 | 1 | $0+\tfrac11(1-0)$ | $1.0$ |
| 2 | 0 | $1+\tfrac12(0-1)$ | $0.5$ |
| 3 | 2 | $0.5+\tfrac13(2-0.5)$ | $1.0$ |
| 4 | 4 | $1.0+\tfrac14(4-1.0)$ | $1.75$ |
| 5 | 3 | $1.75+\tfrac15(3-1.75)$ | $2.0$ |

**Step 3 — Check.** The plain average is $\tfrac{1+0+2+4+3}{5}=\tfrac{10}{5}=2.0 = Q_6$. ✓

**Key concept**

With step size $\tfrac1n$ the incremental rule reproduces the exact running mean using $O(1)$ memory and computation per step — no need to store the reward history. This is the prototype of every value-learning update in the course.

### Exercise 4.2 — Constant step size = exponential recency weighting

For the same reward sequence $1,0,2,4,3$ and $Q_1=0$, use a **constant** step size $\alpha=0.5$:

1. Compute $Q_2,\dots,Q_6$.
2. Show that $Q_6$ is an exponentially weighted average of the rewards; list the weights and verify they (with the initial-estimate weight) sum to $1$.

**Step 1 — Iterate** $Q_{n+1}=Q_n+\alpha(R_n-Q_n)$ with $\alpha=0.5$:

| $n$ | $R_n$ | $Q_{n+1}$ |
|---|---|---|
| 1 | 1 | $0.5$ |
| 2 | 0 | $0.25$ |
| 3 | 2 | $1.125$ |
| 4 | 4 | $2.5625$ |
| 5 | 3 | $2.78125$ |

**Step 2 — Weighted-average form.** Unrolling the recursion gives

$\displaystyle Q_{n+1} = (1-\alpha)^n Q_1 + \sum_{i=1}^{n}\alpha(1-\alpha)^{\,n-i} R_i.$

For $n=5,\ \alpha=0.5$, the weight on $R_i$ is $\alpha(1-\alpha)^{5-i}=0.5\cdot0.5^{5-i}$:

$\displaystyle w(R_5)=0.5,\ w(R_4)=0.25,\ w(R_3)=0.125,\ w(R_2)=0.0625,\ w(R_1)=0.03125,$

plus the weight on the initial estimate $(1-\alpha)^5 = 0.03125$.

**Step 3 — Checks.** Weights sum to $0.5+0.25+0.125+0.0625+0.03125+0.03125 = 1$ ✓. And the weighted sum reproduces $Q_6$:

$\displaystyle 0.5(3)+0.25(4)+0.125(2)+0.0625(0)+0.03125(1)+0.03125(0) = 2.78125 . ✓$

**Key concept**

A constant $\alpha$ makes recent rewards count exponentially more than old ones — ideal for **non-stationary** problems. The price is that the estimate never fully "forgets" the (decaying) influence of the initial guess and never completely converges, continuing to track the latest rewards.

### Exercise 4.3 — Convergence conditions for the step size

The Robbins–Monro conditions guarantee convergence of a stochastic-approximation estimate:

$\displaystyle \text{(C1)}\quad \sum_{n=1}^{\infty}\alpha_n = \infty, \qquad\qquad \text{(C2)}\quad \sum_{n=1}^{\infty}\alpha_n^2 < \infty.$

For each schedule below, state whether (C1) and (C2) hold, and hence whether convergence to the true action value is guaranteed:

$\displaystyle \text{(a)}\ \alpha_n=\tfrac1n, \qquad \text{(b)}\ \alpha_n=\tfrac{1}{n^2}, \qquad \text{(c)}\ \alpha_n=c\ (\text{constant}), \qquad \text{(d)}\ \alpha_n=\tfrac{1}{\sqrt{n}}.$

**Step 1 — Recall the convergence criterion.**

Recall the $\displaystyle p$-series fact: $\displaystyle\sum_n n^{-p}$ **diverges** for $p\le 1$ and **converges** for $p>1$. Check (C1) and (C2) against this fact for each schedule.

**Step 2 — Schedules (a) and (b): the decaying step sizes.**

**(a) $\alpha_n=1/n$.**
- (C1): $\sum 1/n$ is the harmonic series → **diverges** ✓.
- (C2): $\sum 1/n^2$ ($p=2>1$) → **converges** ✓.
- **Both hold → convergence guaranteed.** (This is the sample-average case.)

**(b) $\alpha_n=1/n^2$.**
- (C1): $\sum 1/n^2$ → **converges** ✗ (fails C1).
- Steps shrink too fast; the estimate can get "stuck" before overcoming its initial condition. **Not guaranteed.**

**Step 3 — Schedules (c) and (d): the non-decaying step sizes.**

**(c) $\alpha_n=c$ (constant).**
- (C1): $\sum c=\infty$ ✓.
- (C2): $\sum c^2=\infty$ → **diverges** ✗ (fails C2).
- The estimate never settles; it keeps fluctuating with recent rewards. **Not guaranteed** — but this is exactly what we *want* for non-stationary problems.

**(d) $\alpha_n=1/\sqrt{n}$.**
- (C1): $\sum n^{-1/2}$ ($p=\tfrac12\le1$) → **diverges** ✓.
- (C2): $\sum n^{-1}$ → **diverges** ✗ (fails C2).
- **Not guaranteed.**

**Step 4 — Summary.**

| schedule | C1 ($\sum\alpha_n=\infty$) | C2 ($\sum\alpha_n^2<\infty$) | converges? |
|---|---|---|---|
| $1/n$ | ✓ | ✓ | **yes** |
| $1/n^2$ | ✗ | ✓ | no |
| $c$ | ✓ | ✗ | no (good for non-stationary) |
| $1/\sqrt n$ | ✓ | ✗ | no |

**Key concept**

(C1) keeps the steps *large enough* to overcome initial bias and noise; (C2) makes them *small enough* eventually to settle. Only $\alpha_n=1/n$ satisfies both; constant $\alpha$ deliberately violates (C2) to stay adaptive.

### Exercise 4.4 — $\varepsilon$-greedy action probabilities and expected reward

A $k=4$ armed bandit uses an $\varepsilon$-greedy policy with $\varepsilon=0.2$. The current estimates make action $a_4$ the unique greedy action. The **true** action values are $q(a_1),\dots,q(a_4) = 1,\,2,\,0,\,3$.

1. Compute the probability of selecting each action.
2. Compute the expected reward obtained on a single step under this policy.

**Step 1 — Selection probabilities.** Under $\varepsilon$-greedy the greedy action is chosen either by exploitation (prob. $1-\varepsilon$) or by landing on it during the random draw (prob. $\varepsilon/k$); every other action is chosen only via the random draw:

$\displaystyle \Pr(\text{greedy }a_4) = 1-\varepsilon+\frac{\varepsilon}{k} = 0.8 + \frac{0.2}{4} = 0.8 + 0.05 = 0.85,$
$\displaystyle \Pr(a_1)=\Pr(a_2)=\Pr(a_3) = \frac{\varepsilon}{k} = \frac{0.2}{4} = 0.05.$

Check: $0.85 + 3(0.05) = 1$ ✓.

**Step 2 — Expected reward per step.** $\ \mathbb{E}[R] = \sum_a \Pr(a)\,q(a)$:

$\displaystyle \mathbb{E}[R] = 0.85\,(3) + 0.05\,(1) + 0.05\,(2) + 0.05\,(0) = 2.55 + 0.15 = 2.7.$

(Here $\Pr(\text{greedy})=0.85$ exactly; the $0.85\dots$ that a calculator shows is only floating-point round-off.)

(The greedy action has the highest true value here, so the $0.15$ from exploration is the "cost" paid this step to keep learning.)

**Key concept**

$\varepsilon$-greedy never stops exploring: even after identifying the best action it selects it only with probability $1-\varepsilon+\varepsilon/k$, capping its exploitation. Decaying $\varepsilon$ over time recovers the best of both worlds.

### Exercise 4.5 — Upper Confidence Bound (UCB) selection

A $3$-armed bandit has been played $t=6$ times so far, with selection counts $N=(3,2,1)$ and value estimates $Q=(1.0,\,1.5,\,2.0)$ for arms $1,2,3$. Using the UCB rule with $c=2$,

$\displaystyle A_t=\arg\max_a\Big[\,Q_t(a) + c\sqrt{\tfrac{\ln t}{N_t(a)}}\,\Big],$

compute the UCB score of each arm and determine which arm is selected next. Comment on the role of the bonus.

**Step 1 — Compute the exploration bonus** $c\sqrt{\ln t / N_t(a)}$ with $\ln 6 \approx 1.7918$:

$\displaystyle \text{arm 1: } 2\sqrt{\tfrac{1.7918}{3}} = 2\sqrt{0.5973}=1.5456,\quad \text{arm 2: } 2\sqrt{\tfrac{1.7918}{2}} = 1.8930,\quad \text{arm 3: } 2\sqrt{\tfrac{1.7918}{1}} = 2.6771.$

**Step 2 — Add the value estimate.**

| arm | $Q$ | bonus | UCB score |
|---|---|---|---|
| 1 | 1.0 | 1.5456 | $2.5456$ |
| 2 | 1.5 | 1.8930 | $3.393$ |
| 3 | 2.0 | 2.6771 | $4.6771$ |

**Step 3 — Select.** The maximum UCB score is arm **3** ($4.6771$), which is selected next.

**Step 4 — Interpretation.** Arm 3 wins on *both* counts here: it has the highest estimate **and** the largest bonus (it is the least-tried, so the most uncertain). Even an arm with a lower estimate can be selected if its bonus is large enough — UCB gives the "benefit of the doubt" to under-sampled actions. As an arm is played, its $N$ grows and its bonus shrinks (while $\ln t$ grows only slowly), so attention naturally shifts.

**Key concept**

UCB is *optimism in the face of uncertainty*: it explores deterministically by adding a confidence bonus that is largest for the actions we know least about, rather than exploring blindly like $\varepsilon$-greedy.